In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 
!pip install geopy

# EDA


In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()


# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd

from datetime import date
from tqdm import tqdm
import os 

# Loading Data

In [ ]:
wq_data = pd.read_csv('complete_data.csv')
wq_data.head(3)

# Pre-processing

In [ ]:
wq_data.info()

In [ ]:
print('Duplicates: ', wq_data.duplicated().sum())
print('\n')
print('Missing values:\n', wq_data.isnull().sum())


In [ ]:
# count
# mean
# std
# min
# 25
# 50
# 75
# max

wq_data.describe()

In [ ]:
# Check how common zero runoff is in training data
zero_q = (wq_data['q'] == 0).sum()
total  = len(wq_data)
print(f'Zero runoff samples: {zero_q} ({zero_q/total*100:.1f}%)')

# Check by month to confirm seasonal pattern
wq_data.groupby('Month')['q'].apply(
    lambda x: (x == 0).sum()
).rename('zero_runoff_count').reset_index()

# EDA Section

In [ ]:
wq_data.columns

### Boxplot Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# What's considered as "good" water quality
# Total Alkalinity -> 20 - 200
# Electrical Conductance -> under 800
# Dissolved Reactive Phosphorus -> under 100


def boxplot(df):

    # plot comparison
    fig, ax = plt.subplots(1, 3, figsize=(12,10))
    
    sns.boxplot(data=wq_data['Total Alkalinity'], ax=ax[0])
    ax[0].set_title("Total Alkalinity")
    
    sns.boxplot(data=wq_data['Electrical Conductance'], ax=ax[1])
    ax[1].set_title("Electrical Conductance")
    
    sns.boxplot(data=wq_data['Dissolved Reactive Phosphorus'], ax=ax[2])
    ax[1].set_title("Dissolved Reactive Phosphorus")
    
    plt.tight_layout()
    plt.show()

    return 

boxplot(wq_data)

### Histogram Analysis

In [ ]:
wq_data.columns

In [ ]:
features = ['Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus']
       
def hist(df, cols, row):

    fig, ax = plt.subplots(row,3,figsize=(12,6))
    ax = ax.ravel()

    
    for i, column in enumerate(cols):
        sns.histplot(wq_data, x=column, ax = ax[i], kde=True, bins = 20)
        ax[i].set_title(f"Histogram for {column}")
        
    
    plt.tight_layout()
    plt.show()

    return

hist(wq_data, features,1)

In [ ]:
features_rgb = ['blue',
       'red', 'green', 'nir', 'swir16', 'swir22']

hist(wq_data, features_rgb, 2)

In [ ]:
features_landsat = ['NDMI', 'MNDWI', 'NDVI',
       'NDTI', 'NDBI', 'SAVI']

hist(wq_data, features_landsat, 2)

In [ ]:
features_terraclimate = ['pet', 'ppt', 'tmax', 'def', 'q', 'vpd',
       'ws', 'soil', 'elevation']

hist(wq_data, features_terraclimate, 3)

In [ ]:
wq_data.columns


In [ ]:
df_changed = wq_data.copy()
df_changed.head(3)

In [ ]:
# changing to log - soil, q for highly skewed columns

var = ['q', 'soil', 'ppt']

def change_log(df, var):

    for i, col in enumerate(var):
        df['log_'+col] = np.log1p(df[col].values)

    print(f"Done converting to log values for {var}")

    return df

change_log(df_changed, var)

### Line Plot

In [ ]:

wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'])
df = wq_data.sort_values('Sample Date').set_index('Sample Date')

param = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

# 3. Create subplots correctly
fig, ax = plt.subplots(len(param), 1, figsize=(12, 4*len(param)), sharex=True)

for i, n in enumerate(param):
    
    # 4. Monthly aggregation
    monthly = df[n].resample('MS').mean()

    # 5. Plot (no reset_index needed)
    sns.lineplot(x=monthly.index, y=monthly.values, ax=ax[i])
    
    ax[i].set_title(f"Monthly Mean {n}")
    ax[i].set_ylabel(n)

plt.xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# QUARTERLY mean

fig, ax = plt.subplots(3,1, figsize=(12,4*len(param)))

for i, n in enumerate(param):

    
    quarterly = df[[n]].resample('QE').mean()  
    
    sns.lineplot(data=quarterly, x='Sample Date', y=n, ax=ax[i])
    ax[i].set_title(f"Quarterly Mean {n}")

plt.tight_layout()
plt.show()

In [ ]:
wq_data.columns

# Plotting world map


In [ ]:
wq_data.columns

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# load the default geopandas base map file to plot points on
world = gpd.read_file("ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp")

name_col = 'NAME' if 'NAME' in world.columns else 'name'
sa = world[world[name_col] == "South Africa"]

# Convert your dataframe to GeoDataFrame
gdf = gpd.GeoDataFrame(
    wq_data.copy(),
    geometry=gpd.points_from_xy(wq_data["Longitude"], wq_data["Latitude"]),
    crs="EPSG:4326"
)

# Indicators to plot
indicators = ['Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus']
    

# Create subplots
fig, ax = plt.subplots(3,1, figsize=(14, 10))

for i, n in enumerate(indicators):

    # plot base map
    sa.plot(ax=ax[i], edgecolor="gray", color="ghostwhite", alpha=0.5)
    
    # plot points
    gdf.plot(
        ax=ax[i],
        column=n,
        cmap="viridis",
        legend=True,
        markersize=12,
        alpha=0.7
    )
    
    ax[i].set_title(n)
    ax[i].set_xlabel("Longitude")
    ax[i].set_ylabel("Latitude")
    
plt.tight_layout()
plt.show()

In [ ]:
# Indicators to plot
param = 'elevation'
    

# Create subplots
fig, ax = plt.subplots(figsize=(12, 10))


# plot base map
sa.plot(ax=ax, edgecolor="gray", color="ghostwhite", alpha=0.5)

# plot points
gdf.plot(
    ax=ax,
    column=param,
    cmap="viridis",
    legend=True,
    markersize=12,
    alpha=0.7
)

plt.title("Elevation according to coordinates")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()
plt.show()

## Scatter Plot

In [ ]:
wq_data.columns

In [ ]:
target = ['Total Alkalinity',
        'Electrical Conductance', 'Dissolved Reactive Phosphorus']

features = ['blue',
       'red', 'green', 'nir', 'swir16', 'swir22']
       
def scatterplot(df, row, target, features):

    fig, ax = plt.subplots(row,3,figsize=(16,10))
    ax = ax.ravel()
    
    for i, col in enumerate(features):
        sns.scatterplot(wq_data, x=col, y=target, ax = ax[i])
        ax[i].set_title(f"Histogram for {col}")
        
    plt.tight_layout()
    plt.show()

    return

for i, tar in enumerate(target):

    print('=' *90)
    print(f'Scatter Plot for {tar}')
    print('=' *90)
    scatterplot(wq_data, 2, tar, features)

In [ ]:
features = ['NDMI', 'MNDWI', 'NDVI',
       'NDTI', 'NDBI', 'SAVI']

for i, tar in enumerate(target):

    print('=' *90)
    print(f'Scatter Plot for {tar}')
    print('=' *90)
    scatterplot(wq_data, 2, tar, features)


In [ ]:
features = ['pet', 'ppt', 'tmax', 'def', 'q', 'vpd',
       'ws', 'soil', 'elevation']

for i, tar in enumerate(target):

    print('=' *90)
    print(f'Scatter Plot for {tar}')
    print('=' *90)
    scatterplot(wq_data, 3, tar, features)


In [ ]:
dist = ['dist_johannesburg', 'dist_cape_town']


for i, tar in enumerate(target):

    print('=' *90)
    print(f'Scatter Plot for {tar}')
    print('=' *90)

    
    for n, feats in enumerate(dist):
        
        sns.regplot(wq_data, x=feats, y=tar, ci=None, line_kws={"color": "red"})
        plt.xlabel(feats)
        plt.ylabel(tar)
        plt.title(f'Scatter Plot {feats} vs {tar}')
        plt.show()

## Heatmap 

In [ ]:
wq_data.columns

In [ ]:
cor = wq_data.drop(['Latitude', 'Longitude', 'Sample Date', 
    'Quarter', 'Week_of_year', 'Month', 'blue', 'red', 'aet', 'Week_of_year_sin', 'Week_of_year_cos', 'nir', 'swir16'], axis=1).corr()

plt.figure(figsize=(20,6))
sns.heatmap(cor, annot=True)

In [ ]:
# Spearman corr for nonlinear

spear_corr = wq_data.drop(['Latitude', 'Longitude', 'Sample Date', 
    'Quarter', 'Week_of_year', 'Month', 'blue', 'red', 'aet', 'Week_of_year_sin', 'Week_of_year_cos', 'nir', 'swir16'], axis=1).corr(method = 'spearman')
plt.figure(figsize=(20,6))
sns.heatmap(spear_corr, annot=True)


Features need to be remove due to high correlation to avoid multicollinearity:

swir16
SAVI
day_of_year_sin
day_of_year_cos
week_of_year_sin
week_of_year_cos


### Temporal Analysis: Compare indices across seasons or years to detect trends and anomalies.

In [ ]:
df.index

In [ ]:
indices = ['MNDWI', 'NDTI', 'NDBI',
       'NDVI', 'NDMI', 'SAVI']

fig, ax = plt.subplots(len(indices), 1, figsize=(16, 4*len(indices)), sharex=True)

for i, n in enumerate(indices):
    
    monthly_indices = df[n].resample('MS').mean()
    sns.lineplot(x=monthly.index, y=monthly_indices.values, ax=ax[i])
    
    ax[i].set_title(f"Monthly Mean {n}")
    ax[i].set_ylabel(n)


plt.xlabel("Date")
plt.tight_layout()
plt.show()



In [ ]:
fig, ax = plt.subplots(len(indices), 1, figsize=(16, 4*len(indices)), sharex=True)

for i, n in enumerate(indices):
    
    quarterly_indices = df[n].resample('QE').mean()
    sns.lineplot(x=quarterly_indices.index, y=quarterly_indices.values, ax=ax[i])
    
    ax[i].set_title(f"Quarterly Mean {n}")
    ax[i].set_ylabel(n)


plt.xlabel("Date")
plt.tight_layout()
plt.show()

